# Corrigé : Chapitre 3

*MLSMM2151 Data Science · Séance 2 · Corentin Vande Kerckhove*

Corrigé des sept hack-times des trois notebooks du chapitre 3, dans l'ordre.
Pour chacun : le code commenté, la réponse aux questions d'interprétation, et
les erreurs fréquentes. Les cellules s'exécutent de haut en bas.

In [ ]:
# Cette ligne récupère le fichier utils/donnees.py du dépôt du cours.
!curl -fsSL -o donnees.py https://raw.githubusercontent.com/cvandekerckh/mlsmm2151/main/utils/donnees.py

import pandas as pd
import matplotlib.pyplot as plt
from donnees import charger

In [ ]:
annonces = charger("airbnb_raw")

In [ ]:
# Les conversions des notebooks 1 et 2, dont les solutions ont besoin.
annonces["prix"] = (
    annonces["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)
annonces["superhost"] = annonces["host_is_superhost"].map({"t": 1, "f": 0})
for colonne in ["host_since", "first_review", "last_review"]:
    annonces[colonne] = pd.to_datetime(annonces[colonne], format="%Y-%m-%d")

DATE_INSTANTANE = pd.Timestamp("2025-06-21")

## Notebook 1, hack-time 1 : convertir `host_acceptance_rate`

In [ ]:
# Même geste que pour le taux de réponse : retirer le %, convertir, ramener entre 0 et 1.
annonces["taux_acceptation"] = annonces["host_acceptance_rate"].str.rstrip("%").astype(float) / 100
annonces["taux_acceptation"].describe().round(2)

In [ ]:
print(f"Taux exactement nul : {(annonces['taux_acceptation'] == 0).sum()} annonces")
print(f"Taux manquant       : {annonces['taux_acceptation'].isna().sum()} annonces")

**Lecture.** La médiane est de 0,96 : la plupart des hôtes acceptent presque
toutes les demandes. Mais 344 annonces ont un taux de **0 %**.

**Que peut vouloir dire 0 % ?** Pas forcément un hôte qui refuse tout. Le taux
d'acceptation ne porte que sur les **demandes** de réservation, celles qu'un
hôte doit valider à la main. Une annonce en réservation immédiate n'en reçoit
presque pas, et un hôte qui a reçu une seule demande en un an et l'a laissée
expirer affiche 0 %. Un 0 % calculé sur une demande et un 0 % calculé sur
cent n'ont pas le même poids, et la colonne ne dit pas lequel est lequel. Ce
qu'on peut écrire : *le taux est bas pour 344 annonces, mais le fichier ne
permet pas de savoir sur combien de demandes il est calculé.*

### Erreurs fréquentes

- **`astype(float)` sans retirer le `%`** : `ValueError: could not convert
  string to float: '99%'`. L'erreur est bruyante, tant mieux.
- **Oublier de diviser par 100** : la colonne va de 0 à 100. Rien ne plante,
  mais un modèle ou une comparaison avec `taux_reponse` sera faux. C'est
  l'erreur silencieuse, la plus coûteuse.
- **Remplir les manquants par 0** « pour que ça marche » : les 857 hôtes sans
  taux deviennent des hôtes qui refusent tout.

### Le défi optionnel : Superhosts contre les autres

In [ ]:
annonces.groupby("superhost")["taux_acceptation"].median()

Les Superhosts acceptent davantage : une médiane de 0,97 contre 0,94. L'écart
est net, mais il ne dit pas dans quel sens va la relation : accepter beaucoup
aide-t-il à devenir Superhost, ou les hôtes les plus investis font-ils les deux
à la fois ? Un tableau de médianes constate une association, il ne prouve pas
une cause.

Les 165 hôtes au statut manquant **disparaissent** du calcul : `groupby` ignore
par défaut les clés manquantes. Pour les voir, il faut
`groupby("superhost", dropna=False)`. Encore un filtre silencieux.

## Notebook 1, hack-time 2 : les incohérences de dates


In [ ]:
# 1. De combien de jours l'inscription de l'hôte suit-elle le premier avis ?
avis_avant_hote = annonces["first_review"] < annonces["host_since"]
ecart_jours = (annonces["host_since"] - annonces["first_review"]).dt.days
print(ecart_jours[avis_avant_hote].describe().round(0))

# 2. De combien de jours le dernier avis dépasse-t-il la date du fichier ?
avis_apres_instantane = annonces["last_review"] > DATE_INSTANTANE
depassement = (annonces["last_review"] - DATE_INSTANTANE).dt.days
print(f"\nDépassement maximal : {depassement[avis_apres_instantane].max()} jours")

# 3. Les annonces à vérifier, toutes anomalies confondues.
annonces["date_a_verifier"] = avis_avant_hote | avis_apres_instantane
print(f"Annonces à vérifier : {annonces['date_a_verifier'].sum()}")


**Lecture.** Les écarts de la première anomalie vont de 9 à 2 222 jours,
médiane 268 : des mois, souvent des années. Ce n'est pas une faute de frappe
sur un jour, et ces 22 lignes se concentrent sur 10 comptes. L'explication la
plus plausible est une reprise d'annonces : une agence ouvre un compte et
reprend des logements qui avaient déjà des avis, ou un hôte recrée son compte.
Airbnb garde les avis de l'annonce, mais la date d'inscription est celle du
nouveau compte. **Une réalité, pas une erreur : on garde, et on ne corrige
rien.** C'est l'interprétation « âge de l'hôte » qui est fragile pour ces
comptes, et on le signale.

La seconde anomalie dépasse la date du fichier de 8 jours au plus. Inside
Airbnb ne collecte pas 6 721 pages en une seconde : la collecte s'étale sur
plusieurs jours, et le fichier porte la date de son début. **Une réalité aussi :
on garde.** Supprimer 31 annonces parce que leur dernier avis est trop
**récent** retirerait les plus actives. Selon l'usage, on peut ramener ces
dates au 21 juin, en le justifiant, pour que les durées ne soient pas
négatives.

Au total, 52 annonces portent une date à vérifier, et aucune n'est supprimée.
Dans les deux cas, la décision vient de l'hypothèse sur l'origine de
l'anomalie, pas de l'anomalie elle-même : constater n'est pas décider.

### Erreurs fréquentes

- **Supprimer les lignes « incohérentes » par réflexe.** Pour la seconde
  anomalie, cela retire justement les annonces les plus actives : un biais,
  introduit en croyant nettoyer.
- **Corriger la date d'inscription** en la remplaçant par celle du premier
  avis. C'est inventer une information : on ne sait pas quand l'ancien compte
  a été créé.
- **Ne rien écrire.** Garder les lignes est une décision valable, à condition
  de l'écrire avec sa raison.

### Le défi optionnel : la valeur impossible

Le temps écoulé depuis le dernier avis sera **négatif** pour ces 31 annonces :
leur dernier avis est « dans le futur » par rapport au 21 juin. Rien ne plante,
un nombre négatif est un nombre comme un autre. Personne ne s'en apercevra,
sauf en regardant le minimum de la colonne. C'est ce qu'on vérifie au
notebook 3, hack-time 2.


L'écart va de 9 jours à 2 222 jours, soit plus de six ans. Un écart de six ans
n'est pas une faute de frappe : c'est un historique qui a changé de
propriétaire.

## Notebook 2, hack-time 1 : récupérer `bathrooms` depuis `bathrooms_text`

In [ ]:
# 1. Le premier nombre du texte : "1.5 shared baths" donne 1.5.
nombre = annonces["bathrooms_text"].str.extract(r"(\d+\.?\d*)")[0].astype(float)

# 2. Les demi-salles de bain n'ont pas de chiffre : on leur donne 0,5.
demi = annonces["bathrooms_text"].str.contains("half", case=False, na=False)
nombre = nombre.mask(demi, 0.5)

# 3. On ne complète que ce qui manque : la colonne d'origine reste prioritaire.
annonces["salles_de_bain"] = annonces["bathrooms"].fillna(nombre)

print(f"Manquants avant : {annonces['bathrooms'].isna().sum()}")
print(f"Manquants après : {annonces['salles_de_bain'].isna().sum()}")

De 995 manquants à 4 : les 4 annonces où le nombre **et** le texte manquent.
Celles-là, aucune lecture ne les sauvera.

### Erreurs fréquentes

- **`r"(\d+)"` au lieu de `r"(\d+\.?\d*)"`** : `1.5 baths` devient 1. Rien ne
  plante, la moitié des salles de bain disparaît.
- **Oublier les `half-bath`** : `str.extract` renvoie `NaN` pour eux, et
  l'exercice laisse des manquants qu'il aurait pu combler. La cellule
  ci-dessous les compte.
- **Oublier `[0]`** : `str.extract` renvoie un `DataFrame`, pas une `Series`,
  et `fillna` avec un `DataFrame` ne fait pas ce qu'on croit.
- **Oublier `na=False` dans `str.contains`** : les 5 textes manquants donnent
  un masque avec des `NaN`, et `mask` refuse de s'en servir.

In [ ]:
# Ce qu'on obtient en oubliant les demi-salles de bain.
sans_demi = annonces["bathrooms"].fillna(annonces["bathrooms_text"].str.extract(r"(\d+\.?\d*)")[0].astype(float))
print(f"Manquants restants sans traiter les half-bath : {sans_demi.isna().sum()}")

### Le défi optionnel : l'extraction est-elle fidèle ?

In [ ]:
deux_remplis = annonces["bathrooms"].notna() & nombre.notna()
identiques = annonces.loc[deux_remplis, "bathrooms"] == nombre[deux_remplis]
print(f"{identiques.mean():.0%} d'accord sur {deux_remplis.sum()} annonces")

Accord parfait. Inside Airbnb construit vraisemblablement `bathrooms` **à
partir** de `bathrooms_text`, et ne l'a simplement pas fait pour toutes les
lignes. Le manque de `bathrooms` est donc un artefact de traitement, pas une
information absente. Imputer une médiane aurait remplacé une valeur connue par
une valeur inventée.

**L'information perdue** : le texte dit si la salle de bain est **partagée**
ou **privée**. `1 bath` et `1 shared bath` deviennent tous deux 1, alors que
pour un voyageur ce n'est pas du tout la même chose. Elle mériterait sa propre
colonne, par exemple `str.contains("shared", case=False)`. C'est du *feature
engineering*, au chapitre 7.

## Notebook 2, hack-time 2 : ce que le dédoublonnage change


In [ ]:
cle = ["host_id", "name"]
dedoublonnees = annonces.drop_duplicates(subset=cle)

# 2. Les logements de Molenbeek-Saint-Jean, avant et après.
molenbeek = annonces["neighbourhood_cleansed"] == "Molenbeek-Saint-Jean"
molenbeek_dedoublonne = dedoublonnees["neighbourhood_cleansed"] == "Molenbeek-Saint-Jean"
print(f"Molenbeek : {molenbeek.sum()} annonces, {molenbeek_dedoublonne.sum()} après dédoublonnage")


# 3. La part des annonces tenue par les hôtes de plus de dix annonces.
def part_grands_hotes(table):
    par_hote = table["host_id"].value_counts()
    return par_hote[par_hote > 10].sum() / len(table)


print(f"Part des grands hôtes : {part_grands_hotes(annonces):.1%} avant, "
      f"{part_grands_hotes(dedoublonnees):.1%} après dédoublonnage")


**Lecture.** Molenbeek passe de 235 à 227 annonces, et la part des grands
hôtes de 13,1 % à 12,2 %. Dans les deux cas le dédoublonnage change la
réponse, et dans les deux cas c'est une erreur, mais pas pour la même raison.

**Combien de logements à Molenbeek ?** Les dix « Luxurious studio lively
area » sont dix chambres distinctes, avec des capacités, des avis et des
positions différents. Chacune est un logement offert à la location : les
retirer sous-estime l'offre.

**La part des grands hôtes ?** C'est précisément l'information recherchée : un
hôte qui publie dix chambres sous le même titre est un hôte professionnel, et
le dédoublonnage efface le phénomène qu'on veut mesurer.

**Le principe** : un quasi-doublon n'est ni bon ni mauvais en soi. Il dépend
de ce qu'est une ligne pour votre question : un logement, une observation pour
un modèle, ou un morceau d'activité d'un hôte. Pour un modèle de prix, par
exemple, dix lignes presque identiques pèsent dix fois dans l'apprentissage,
et si certaines tombent dans l'entraînement et d'autres dans le test, le score
sera trop optimiste : c'est une forme de fuite de données, l'objet du
chapitre 4.

### Erreurs fréquentes

- **`drop_duplicates(subset=["host_id", "name"])` par défaut**, « parce qu'un
  doublon c'est mal ». On supprime 112 logements réels, sans les regarder.
- **Conclure « pas de doublons » après `duplicated()`** : zéro doublon exact
  ne dit rien des quasi-doublons.

### Le défi optionnel : le point final


In [ ]:
colonnes = ["host_id", "name", "room_type", "neighbourhood_cleansed", "number_of_reviews"]
annonces.loc[annonces["name"].str.contains("Luxurious studio lively area", regex=False), colonnes]

L'annonce « Luxurious studio lively area. » appartient au même hôte, mais un
point final suffit à la faire échapper à la règle « même titre ». Une
correspondance exacte est fragile face à la ponctuation, aux majuscules et
aux espaces.

Pour la rendre plus robuste, on **normalise** le titre avant de comparer :
minuscules, ponctuation et espaces superflus retirés, par exemple
`name.str.lower().str.replace(r"[^\w\s]", "", regex=True).str.strip()`. Le
risque inverse : à force de simplifier, deux annonces réellement différentes
deviennent « identiques ». « Studio Louise » et « Studio Louise 2 » ne sont
pas le même logement. Plus la règle est souple, plus il faut regarder les
groupes qu'elle forme.

## Notebook 2, hack-time 3 : la règle IQR sur le prix

In [ ]:
prix = annonces["prix"].dropna()

q1 = prix.quantile(0.25)
q3 = prix.quantile(0.75)
iqr = q3 - q1
borne_basse = q1 - 1.5 * iqr
borne_haute = q3 + 1.5 * iqr

print(f"Q1 = {q1}, Q3 = {q3}, IQR = {iqr}")
print(f"Bornes : [{borne_basse}, {borne_haute}]")

In [ ]:
hors_iqr = (prix < borne_basse) | (prix > borne_haute)
print(f"Règle IQR          : {hors_iqr.sum()} annonces écartées sur {len(prix)}")
print(f"Règle métier       : {(~prix.between(20, 1000)).sum()} annonces écartées")
print(f"Sous la borne basse : {(prix < borne_basse).sum()}")

**Lecture.** La règle IQR écarte 391 annonces, la règle métier 15. La règle
IQR place sa borne haute à 205 par nuit, et sa borne basse à -19, qui ne
peut écarter aucun prix. Elle ne coupe donc que d'un côté.

**Ce que la règle IQR écarte et que la règle métier garde** : tout logement
au-dessus de 205 par nuit. C'est-à-dire les grands appartements, les maisons
pour huit ou dix personnes, les logements de standing. Ce ne sont pas des
erreurs : c'est le haut du marché. Un modèle de prix entraîné sans eux serait
incapable d'estimer le prix d'un grand logement.

La règle IQR suppose implicitement une distribution à peu près symétrique.
Les prix ne le sont pas, et la règle prend la queue de la distribution pour
des anomalies. La règle métier, elle, dit ce qu'on cherche : écarter les prix
qui ne peuvent pas être de vrais prix de nuitée, comme 9 999, et les prix
symboliques sous 20.

### Erreurs fréquentes

- **Appliquer la règle IQR par défaut**, parce qu'elle est « objective ».
  Elle ne l'est pas : le 1,5 est une convention, et la règle ignore tout de
  la question posée.
- **Calculer les quantiles sur `annonces["prix"]` sans `dropna()`** :
  `quantile` ignore les manquants, mais le compte des lignes « hors bornes »
  doit se faire sur les seules annonces qui ont un prix, sans quoi les
  proportions sont fausses.
- **Oublier de vérifier ce qu'on écarte.** Une règle se juge en regardant
  quelques lignes qu'elle supprime.

### Le défi optionnel : la règle IQR sur le logarithme

In [ ]:
import numpy as np

log_prix = np.log(prix)
q1_log, q3_log = log_prix.quantile([0.25, 0.75])
iqr_log = q3_log - q1_log
basse_log, haute_log = q1_log - 1.5 * iqr_log, q3_log + 1.5 * iqr_log

hors_iqr_log = (log_prix < basse_log) | (log_prix > haute_log)
print(f"Bornes ramenées en euros : [{np.exp(basse_log):.0f}, {np.exp(haute_log):.0f}]")
print(f"Annonces écartées        : {hors_iqr_log.sum()}, dont {(log_prix < basse_log).sum()} par le bas")

Sur le logarithme, les bornes deviennent environ [26, 307] et la règle écarte
185 annonces, dont 41 par le bas : des deux côtés, cette fois. Le logarithme rend la distribution
plus symétrique : la règle redevient sensible aux prix anormalement **bas**,
et moins sévère avec le haut du marché. Elle reste plus sévère que la règle
métier, qui garde tout ce qui va de 20 à 1 000.

## Notebook 3, hack-time 1 : l'ancienneté de l'annonce

In [ ]:
# Même calcul que l'ancienneté de l'hôte, à partir du premier avis.
annonces["anciennete_annonce_annees"] = (DATE_INSTANTANE - annonces["first_review"]).dt.days / 365.25
annonces["anciennete_annonce_annees"].describe().round(2)

In [ ]:
print(f"Annonces de moins d'un an : {(annonces['anciennete_annonce_annees'] < 1).sum()}")
print(f"Manquants                 : {annonces['anciennete_annonce_annees'].isna().sum()}")

**Lecture.** L'annonce médiane a 1,86 an, et 1 937 annonces ont moins d'un an.

**Les 925 manquants** sont les annonces sans avis, celles du notebook 2 :
pas d'avis, donc pas de premier avis, donc pas d'âge calculable. L'âge
calculé est d'ailleurs celui du **premier avis**, pas celui de la mise en
ligne, que le fichier ne contient pas. Une annonce publiée en 2019 et réservée
pour la première fois en 2024 aura un an.

### Erreurs fréquentes

- **Diviser par 365 au lieu de 365,25** : l'écart est faible, mais votre
  colonne ne correspondra plus exactement à celle d'`airbnb_train`.
- **Oublier `.dt.days`** : diviser un écart de dates (`Timedelta`) par un
  nombre donne encore un `Timedelta`, pas un nombre d'années.
- **Lire l'âge du premier avis comme l'âge de l'annonce** sans le dire.

### Le défi optionnel : des annonces plus jeunes que leurs hôtes

In [ ]:
annonces["anciennete_hote_annees"] = (DATE_INSTANTANE - annonces["host_since"]).dt.days / 365.25
annonces[["anciennete_hote_annees", "anciennete_annonce_annees"]].median().round(2)

7,95 ans pour les hôtes, 1,86 an pour les annonces. Deux explications, qui
s'additionnent :

- **un hôte vit plus longtemps que ses annonces** : il s'inscrit une fois,
  puis ouvre, ferme et remplace des annonces au fil des ans ;
- **le biais de survie** : les annonces anciennes qui ont fermé ne sont plus
  dans le fichier, alors que leur hôte, s'il a encore une annonce active, y
  est toujours, avec sa date d'inscription d'origine.

À quoi s'ajoute le décalage entre la mise en ligne et le premier avis, qui
rajeunit artificiellement les annonces.

## Notebook 3, hack-time 2 : pourquoi une date fixe

Les deux cellules ci-dessous font le calcul. La lecture suit.


In [ ]:
avec_today = (pd.Timestamp.today() - annonces["host_since"]).dt.days / 365.25
print(f"Médiane avec DATE_INSTANTANE : {annonces['anciennete_hote_annees'].median():.2f} ans")
print(f"Médiane avec today()         : {avec_today.median():.2f} ans (change chaque jour)")

In [ ]:
annonces["derniere_review_annees"] = (DATE_INSTANTANE - annonces["last_review"]).dt.days / 365.25
print(f"Minimum : {annonces['derniere_review_annees'].min():.3f} an")
print(f"Valeurs négatives : {(annonces['derniere_review_annees'] < 0).sum()}")

**1. Avec `pd.Timestamp.today()`.** La médiane de l'ancienneté des hôtes
augmente du temps écoulé depuis le 21 juin 2025 : environ 1,4 an de plus le
jour de la séance 2, un mois de plus encore si vous relancez le notebook dans
un mois. **Le même code sur les mêmes données donne un résultat différent
chaque jour.** Le notebook n'est plus reproductible : un correcteur qui le
relance n'obtient pas vos chiffres. La date de référence est celle des
données, pas celle du calcul, et elle s'écrit dans le code.

**2. Les durées négatives.** Le minimum vaut -0,022 an, soit les 8 jours de
dépassement vus au notebook 1, pour 31 annonces. Rien n'a planté : une durée
négative est un nombre comme un autre, et seul le minimum la révèle. On ne
change pas de date de référence, ce qui fausserait toutes les autres durées :
on traite les 31 lignes, par exemple en ramenant leur durée à 0, et on
l'écrit.

### Erreurs fréquentes

- **`today()` dans un notebook rendu** : il tourne, et donne d'autres chiffres
  chez le correcteur. C'est un défaut de reproductibilité, noté comme tel.
- **Ne pas regarder le minimum d'une durée.** Une durée négative ne lève
  aucune erreur.

### Le défi optionnel : le modèle utilisé en juin 2027

Les données décrivent le marché du 21 juin 2025. Calculer leurs durées par
rapport à novembre 2026 attribue à chaque hôte une ancienneté qu'il n'avait
pas au moment de la photographie : le modèle apprend une relation entre le
prix de juin 2025 et des anciennetés gonflées d'un an et demi. En production,
il faudra calculer les anciennetés des nouvelles annonces exactement de la
même façon, par rapport à la date où leurs données ont été collectées.


Le minimum vaut -0,022 an, soit 8 jours : l'avis du 29 juin. Et 31 valeurs
négatives, exactement les 31 avis tardifs du notebook 1. Une anomalie
constatée dans le premier notebook a produit une valeur impossible dans le
troisième. C'est pour cela qu'on les cherche, et qu'on écrit ce qu'on en fait.

Pour mémoire, la colonne `derniere_review_annees` d'`airbnb_train` a été
construite sans ce traitement : vous y retrouverez ces valeurs légèrement
négatives.